# LeetCode 466 — Count The Repetitions (Hard) bằng Python

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def getMaxRepetitions(self, s1: str, n1: int, s2: str, n2: int) -> int:

```

**Mục tiêu sau khi học xong:**

1. Hiểu chính xác hai định nghĩa của đề: chuỗi lặp **`[s, n]`** và khái niệm **"thu được" (obtained)** — thực chất là **subsequence (dãy con)**.
2. Biết kiểm tra subsequence bằng **hai con trỏ (two pointers)** và hiểu **vì sao khớp tham lam (greedy) — khớp càng sớm càng tốt — không bao giờ thiệt**.
3. Biết quy bài toán về một con số duy nhất: **`K` = số bản `s2` ghép liền nhau tối đa** lấy được từ `s1 * n1`, rồi **đáp án = `K // n2`**.
4. Viết **Cách 1 — brute force** (xây thật chuỗi `s1 * n1`) làm "trọng tài", và thấy nó vỡ ở đâu khi `n1 = 10^6`.
5. Viết **Cách 2 — mô phỏng từng khối `s1`** không cần xây chuỗi, chỉ giữ hai biến `j` và `count`.
6. Học kỹ thuật **phát hiện chu kỳ (cycle detection)** từ căn bản, rồi viết **Cách 3 — nhảy qua chu kỳ** (cách chính), trace kỹ ví dụ `"acb", 4, "ab", 2`.
7. Viết **Cách 4 — bảng tiền xử lý `next_j / gain`** cho mỗi vị trí bắt đầu, hiểu bài toán như "đi trên một hàm".
8. Nộp bản `class Solution`, kiểm thử chéo mọi cách với brute force, và tự giải các biến thể.

## 1. Đề bài

Ta định nghĩa `str = [s, n]` là chuỗi `str` gồm chuỗi `s` **nối liền nhau `n` lần**.

- Ví dụ: `str == ["abc", 3] == "abcabcabc"`.

Ta định nghĩa chuỗi `s1` **có thể thu được (can be obtained)** từ chuỗi `s2` nếu ta có thể **xóa một số ký tự** của `s2` sao cho nó trở thành `s1`.

- Ví dụ: `s1 = "abc"` có thể thu được từ `s2 = "abdbec"` bằng cách xóa các ký tự `d`, `b` (thứ hai), `e` (trên LeetCode chúng được in đậm, gạch chân).

Cho hai chuỗi `s1`, `s2` và hai số nguyên `n1`, `n2`. Ta có hai chuỗi `str1 = [s1, n1]` và `str2 = [s2, n2]`.

Hãy trả về **số nguyên `m` lớn nhất** sao cho `str = [str2, m]` có thể thu được từ `str1`.

**Ví dụ 1**

```text
Input: s1 = "acb", n1 = 4, s2 = "ab", n2 = 2
Output: 2
```

**Ví dụ 2**

```text
Input: s1 = "acb", n1 = 1, s2 = "acb", n2 = 1
Output: 1
```

**Constraints (ràng buộc):**

- `1 <= s1.length, s2.length <= 100`
- `s1` và `s2` chỉ gồm chữ cái tiếng Anh viết thường (lowercase English letters).
- `1 <= n1, n2 <= 10^6`

> Hai con số cần nhớ ngay: chuỗi gốc ngắn (**≤ 100**) nhưng số lần lặp **rất lớn (tới 10^6)**. Nếu xây thật `s1 * n1` thì có thể dài tới `100 × 10^6 = 10^8` ký tự. Toàn bộ cái khó của bài nằm ở chỗ: **làm sao "đếm" trên một chuỗi khổng lồ mà không đi hết nó**.

In [1]:
s1, n1, s2, n2 = "acb", 4, "ab", 2

str1 = s1 * n1          # [s1, n1]
str2 = s2 * n2          # [s2, n2]
print("str1 = [s1, n1] =", str1, "| dài", len(str1))
print("str2 = [s2, n2] =", str2, "| dài", len(str2))

for m in range(1, 4):
    print(f"[str2, {m}] =", str2 * m)

str1 = [s1, n1] = acbacbacbacb | dài 12
str2 = [s2, n2] = abab | dài 4
[str2, 1] = abab
[str2, 2] = abababab
[str2, 3] = abababababab


### Đọc kỹ Ví dụ 1 bằng tay

`str1 = "acbacbacbacb"` (4 khối `"acb"`), `str2 = "abab"`.

Câu hỏi: lặp `str2` bao nhiêu lần (`m` lớn nhất) mà vẫn **xóa ký tự** của `str1` để được nó?

- `m = 1`: cần `"abab"`. Khối 1 cho `a…b`, khối 2 cho `a…b` → được. ✔
- `m = 2`: cần `"abababab"` = 4 cặp `ab`. Mỗi khối `"acb"` cho đúng một cặp `a…b` (xóa `c`), có 4 khối → vừa đủ 4 cặp. ✔
- `m = 3`: cần 6 cặp `ab`, nhưng `str1` chỉ có 4 chữ `a` → không thể. ✘

Vậy đáp án là `2`.

**Ví dụ 2**: `str1 = "acb"`, `str2 = "acb"`. `m = 1` thì `[str2, 1] = "acb"` thu được (không cần xóa gì). `m = 2` cần `"acbacb"` dài 6 > 3 → không thể. Đáp án `1`.

Để ý cách mình vừa nghĩ: mình **đếm xem `str1` chứa được bao nhiêu bản `s2 = "ab"` nối nhau** (4 bản), rồi chia cho số bản `s2` trong một `str2` (2 bản) → `4 // 2 = 2`. Đây sẽ là xương sống của mọi lời giải phía sau.

## 2. Kiến thức nền 1: nhân chuỗi `s * n` và ký hiệu `[s, n]`

Trong Python, `s * n` (với `s` là chuỗi, `n` là số nguyên) tạo ra chuỗi mới gồm `s` lặp lại `n` lần. Đây đúng là `[s, n]` của đề.

Một tính chất nhỏ nhưng **rất quan trọng** cho bài này:

```text
[[s, a], b] = [s, a * b]
```

Ví dụ: `[["ab", 2], 3] = ["abab", 3] = "abababababab" = ["ab", 6]`.

Áp dụng vào đề: `[str2, m] = [[s2, n2], m] = [s2, n2 * m]`. Tức là "lặp `str2` `m` lần" **chính là** "lặp `s2` đúng `n2 * m` lần". Nhờ vậy ta chỉ cần làm việc với `s2`, không cần quan tâm `str2` nữa.

In [2]:
s = "ab"
print(repr(s * 3))                 # 'ababab'
print(len(s * 3) == len(s) * 3)    # độ dài nhân lên đúng bằng số lần lặp
print(repr(s * 0))                 # '' : lặp 0 lần là chuỗi rỗng

# [[s, a], b] == [s, a*b]
a, b = 2, 3
print((s * a) * b == s * (a * b), "->", (s * a) * b)

'ababab'
True
''
True -> abababababab


## 3. Kiến thức nền 2: "thu được" = subsequence (dãy con)

Đề nói "`s1` thu được từ `s2` nếu xóa một số ký tự của `s2` để được `s1`". Trong tin học, điều này có tên là:

> **`s1` là subsequence (dãy con) của `s2`**: xóa đi **một số** ký tự (có thể 0 ký tự) của `s2`, **giữ nguyên thứ tự** các ký tự còn lại, ta được `s1`.

Phân biệt với **substring (chuỗi con liên tiếp)**:

| | Substring | Subsequence |
|---|---|---|
| Phải liên tiếp? | Có | Không |
| `"ac"` trong `"abc"`? | Không | Có (xóa `b`) |
| `"ca"` trong `"abc"`? | Không | **Không** (thứ tự bị đảo) |
| Toán tử Python | `"bc" in "abc"` | không có sẵn, phải tự viết |

Ví dụ đề: `"abc"` từ `"abdbec"`:

```text
chỉ số:  0 1 2 3 4 5
s2    :  a b d b e c
giữ   :  ^ ^       ^      -> "a" "b" "c"  (xóa d, b, e)
```

Cái bẫy đầu tiên của người mới: dùng `in` của Python. `"abc" in "abdbec"` trả về `False` vì `in` kiểm tra **substring**, không phải subsequence.

In [3]:
s2_demo = "abdbec"
keep = [0, 1, 5]                       # các chỉ số được giữ lại
kept = "".join(s2_demo[i] for i in keep)
print("Giữ chỉ số", keep, "->", repr(kept))

deleted = [s2_demo[i] for i in range(len(s2_demo)) if i not in keep]
print("Các ký tự bị xóa:", deleted)

print("'abc' in 'abdbec' ?", "abc" in s2_demo, " <- in là kiểm tra SUBSTRING, không dùng được")

Giữ chỉ số [0, 1, 5] -> 'abc'
Các ký tự bị xóa: ['d', 'b', 'e']
'abc' in 'abdbec' ? False  <- in là kiểm tra SUBSTRING, không dùng được


## 4. Kiểm tra subsequence bằng hai con trỏ (two pointers)

**Two pointers** nghĩa là dùng hai biến chỉ số chạy trên hai chuỗi:

- `i` chạy trên chuỗi dài `text` (chuỗi ta được phép xóa ký tự).
- `j` chỉ vào **ký tự tiếp theo cần tìm** trong chuỗi `pattern`.

Quy tắc rất đơn giản: duyệt từng ký tự `text[i]`:

- Nếu `text[i] == pattern[j]` → "dùng" ký tự này, tăng `j` lên 1 (chuyển sang tìm ký tự kế tiếp của `pattern`).
- Nếu không bằng → coi như **xóa** `text[i]`, `j` giữ nguyên.

Kết thúc: nếu `j == len(pattern)` (đã tìm đủ mọi ký tự) thì `pattern` là subsequence của `text`.

Đây là cách khớp **tham lam (greedy)**: hễ gặp ký tự khớp là dùng ngay, không chờ ký tự khớp phía sau.

In [4]:
def is_subsequence(pattern: str, text: str) -> bool:
    j = 0                                  # ký tự tiếp theo cần tìm trong pattern
    for ch in text:
        if j < len(pattern) and ch == pattern[j]:
            j += 1                         # khớp -> chuyển sang ký tự kế tiếp
    return j == len(pattern)


print(is_subsequence("abc", "abdbec"))   # True
print(is_subsequence("ac", "abc"))       # True
print(is_subsequence("ca", "abc"))       # False: thứ tự bị đảo
print(is_subsequence("", "abc"))         # True: chuỗi rỗng luôn thu được
print(is_subsequence("abab", "acbacb"))  # True

True
True
False
True
True


### Trace hai con trỏ: `pattern = "abc"`, `text = "abdbec"`

| i | `text[i]` | `pattern[j]` cần tìm | Khớp? | `j` sau |
|---:|:---:|:---:|:---:|---:|
| 0 | `a` | `a` | có | 1 |
| 1 | `b` | `b` | có | 2 |
| 2 | `d` | `c` | không (xóa) | 2 |
| 3 | `b` | `c` | không (xóa) | 2 |
| 4 | `e` | `c` | không (xóa) | 2 |
| 5 | `c` | `c` | có | 3 = len → xong |

Cell dưới in đúng bảng này bằng code.

In [5]:
def trace_is_subsequence(pattern, text):
    j = 0
    for i, ch in enumerate(text):
        need = pattern[j] if j < len(pattern) else "-"
        if j < len(pattern) and ch == pattern[j]:
            j += 1
            action = "khớp"
        else:
            action = "xóa"
        print(f"i={i} text[i]={ch!r} cần={need!r} -> {action:4s} j={j}")
    print("Kết luận:", j == len(pattern))


trace_is_subsequence("abc", "abdbec")

i=0 text[i]='a' cần='a' -> khớp j=1
i=1 text[i]='b' cần='b' -> khớp j=2
i=2 text[i]='d' cần='c' -> xóa  j=2
i=3 text[i]='b' cần='c' -> xóa  j=2
i=4 text[i]='e' cần='c' -> xóa  j=2
i=5 text[i]='c' cần='c' -> khớp j=3
Kết luận: True


### Vì sao khớp tham lam (khớp sớm nhất) không bao giờ thiệt?

Câu hỏi tự nhiên: khi `text[i]` khớp `pattern[j]`, liệu có lúc **bỏ qua** nó để dùng một ký tự giống hệt ở xa hơn lại tốt hơn không?

**Không.** Lý luận "đổi chỗ" (exchange argument):

- Giả sử có một cách khớp hợp lệ nào đó dùng vị trí `i'` > `i` cho `pattern[j]`.
- Ta đổi sang dùng vị trí `i` (sớm hơn, cùng ký tự). Mọi ký tự `pattern[j+1], pattern[j+2], ...` vẫn được khớp ở các vị trí cũ (vốn nằm sau `i'`, nên càng nằm sau `i`).
- Vậy cách khớp mới vẫn hợp lệ, và phần còn lại của `text` để dùng cho các ký tự sau **không ít đi**.

Nói gọn: **khớp càng sớm, phần đuôi còn lại càng dài, càng nhiều cơ hội**. Ý này cực kỳ quan trọng: nó cho phép ta **đếm bằng một lần duyệt** mà không phải thử mọi cách xóa (có `2^len` cách!).

## 5. Đếm số bản `s2` ghép liền nhau: con trỏ `j` "quay vòng"

Bài toán thật không hỏi "`s2` có phải subsequence không", mà hỏi: **`s2` lặp lại tối đa bao nhiêu lần** (`s2 s2 s2 ...`) mà vẫn là subsequence của `text`?

Mẹo: vẫn dùng một con trỏ `j` trên `s2`, nhưng khi `j` chạm `len(s2)` (vừa ghép xong một bản `s2`), ta:

- tăng bộ đếm `count += 1`,
- đưa `j` về `0` để bắt đầu tìm bản `s2` tiếp theo **ngay từ ký tự kế tiếp của `text`**.

Vì `s2 * K` chỉ là `s2` nối với chính nó, nên "khớp tham lam `s2 * K`" giống hệt "khớp tham lam `s2`, xong thì quay lại đầu `s2` và khớp tiếp". Theo mục 4, cách tham lam cho **`count` lớn nhất có thể**. Ta gọi con số này là **`K`**.

Ví dụ: `text = "aabbab"`, `s2 = "ab"`:

```text
a(khớp a) a(xóa) b(khớp b -> xong bản 1) b(xóa) a(khớp a) b(khớp b -> xong bản 2)
```

→ `K = 2`. Kiểm tra: `"abab"` là subsequence của `"aabbab"`, còn `"ababab"` (cần 3 chữ `a`) thì không.

In [6]:
def count_copies(text: str, s2: str) -> int:
    # Số bản s2 ghép liền nhau tối đa là subsequence của text (khớp tham lam)
    j = 0
    count = 0
    for ch in text:
        if ch == s2[j]:
            j += 1
            if j == len(s2):       # vừa ghép xong một bản s2
                count += 1
                j = 0              # bắt đầu bản tiếp theo
    return count


print(count_copies("aabbab", "ab"))         # 2
print(count_copies("acbacbacbacb", "ab"))   # 4  (Ví dụ 1)
print(count_copies("acb", "acb"))           # 1  (Ví dụ 2)
print(count_copies("abc", "d"))             # 0  ('d' không có trong text)

# đối chiếu với định nghĩa: s2*K thu được, s2*(K+1) thì không
K = count_copies("aabbab", "ab")
print(is_subsequence("ab" * K, "aabbab"), is_subsequence("ab" * (K + 1), "aabbab"))

2
4
1
0
True False


## 6. Từ `K` suy ra đáp án: `m = K // n2`

Nhắc lại mục 2: `[str2, m] = [s2, n2 * m]`. Vậy câu hỏi "`[str2, m]` có thu được từ `str1` không?" tương đương "`s2` lặp `n2 * m` lần có thu được không?".

- Nếu `s2 * K` thu được thì `s2 * K'` với mọi `K' ≤ K` cũng thu được (cứ xóa bớt phần đuôi). Tính chất này gọi là **đơn điệu (monotonic)**.
- `K` là số bản lớn nhất thu được. Vậy `[str2, m]` thu được **khi và chỉ khi** `n2 * m ≤ K`.
- `m` lớn nhất thỏa `n2 * m ≤ K` là `m = K // n2` (phép **chia lấy phần nguyên** `//`).

Ví dụ 1: `K = 4`, `n2 = 2` → `m = 4 // 2 = 2`. Nếu `n2 = 3` thì `m = 4 // 3 = 1` (lặp `"ababab"` một lần cần 3 bản `ab` ≤ 4, hai lần cần 6 > 4).

Cell dưới đối chiếu công thức `K // n2` với **định nghĩa nguyên văn của đề** (thử từng `m` bằng `is_subsequence`).

In [7]:
def max_m_by_definition(s1, n1, s2, n2):
    # Đúng nguyên văn đề: tìm m lớn nhất để [str2, m] là subsequence của str1
    str1, str2 = s1 * n1, s2 * n2
    m = 0
    while is_subsequence(str2 * (m + 1), str1):
        m += 1
    return m


s1, n1, s2 = "acb", 4, "ab"
K = count_copies(s1 * n1, s2)
for n2 in [1, 2, 3, 4, 5]:
    print(f"n2={n2}: K // n2 = {K} // {n2} = {K // n2} | theo định nghĩa: {max_m_by_definition(s1, n1, s2, n2)}")

n2=1: K // n2 = 4 // 1 = 4 | theo định nghĩa: 4
n2=2: K // n2 = 4 // 2 = 2 | theo định nghĩa: 2
n2=3: K // n2 = 4 // 3 = 1 | theo định nghĩa: 1
n2=4: K // n2 = 4 // 4 = 1 | theo định nghĩa: 1
n2=5: K // n2 = 4 // 5 = 0 | theo định nghĩa: 0


> **Tóm tắt phần nền:** bài toán quy về đúng một việc — tính **`K` = số bản `s2` khớp tham lam được trên `[s1, n1]`**, rồi trả `K // n2`. Mọi cách giải phía sau chỉ khác nhau ở **cách tính `K` nhanh đến đâu**.

## 7. Cách 1 — Brute force: xây thật chuỗi `s1 * n1`

Ý tưởng người mới nghĩ ra đầu tiên: cứ làm đúng như đề nói.

1. Xây `str1 = s1 * n1`.
2. Duyệt `str1` bằng con trỏ `j` quay vòng trên `s2` (mục 5) để đếm `K`.
3. Trả `K // n2`.

Cách này **chắc chắn đúng** (chỉ là ghép các mảnh đã chứng minh ở trên) và rất ngắn. Ta sẽ dùng nó làm **"trọng tài"**: mọi cách nhanh hơn phía sau đều phải cho kết quả giống nó trên các input nhỏ.

In [8]:
def repetitions_bruteforce(s1: str, n1: int, s2: str, n2: int) -> int:
    str1 = s1 * n1                 # xây thật chuỗi khổng lồ
    j = 0
    count = 0                      # K: số bản s2 đã ghép xong
    for ch in str1:
        if ch == s2[j]:
            j += 1
            if j == len(s2):
                count += 1
                j = 0
    return count // n2


print(repetitions_bruteforce("acb", 4, "ab", 2))    # 2  (Ví dụ 1)
print(repetitions_bruteforce("acb", 1, "acb", 1))   # 1  (Ví dụ 2)
print(repetitions_bruteforce("aaa", 3, "aa", 1))    # 9 chữ 'a' -> 4 bản "aa"
print(repetitions_bruteforce("abc", 5, "d", 1))     # 0  ('d' không có trong s1)

2
1
4
0


### Nút thắt của Cách 1

Gọi `L1 = len(s1)`, `L2 = len(s2)`.

- **Thời gian**: duyệt `n1 · L1` ký tự → tệ nhất `10^6 × 100 = 10^8` bước. Một vòng `for` Python chỉ chạy cỡ vài chục triệu bước đơn giản mỗi giây, nên `10^8` bước mất **nhiều giây** (tùy máy, có khi tới hàng chục giây) — vượt xa giới hạn thời gian, nhất là khi đây mới là một test.
- **Bộ nhớ**: chuỗi `s1 * n1` dài tới `10^8` ký tự ≈ **100 MB** chỉ để chứa một chuỗi mà ta đọc lướt qua đúng một lần.

Cell dưới đo thử trên chuỗi `10^6` ký tự rồi nhân lên để ước lượng (con số cụ thể tùy máy; điều đáng chú ý là thời gian tăng **tuyến tính** theo `n1`: tăng `n1` lên 100 lần thì chậm đi khoảng 100 lần).

In [9]:
import time

s1_big, s2_big = "abcdefghij" * 10, "aceg" * 25        # L1 = 100, L2 = 100
n1_small = 10_000                                     # 10^6 ký tự
t0 = time.perf_counter()
repetitions_bruteforce(s1_big, n1_small, s2_big, 1)
dt = time.perf_counter() - t0

print(f"n1 = {n1_small:>9,}: {dt:.3f} giây, chuỗi dài {len(s1_big) * n1_small:,} ký tự")
print(f"n1 = {10**6:>9,}: ước lượng ~{dt * 100:.0f} giây, chuỗi dài {len(s1_big) * 10**6:,} ký tự (~100 MB)")

n1 =    10,000: 0.041 giây, chuỗi dài 1,000,000 ký tự
n1 = 1,000,000: ước lượng ~4 giây, chuỗi dài 100,000,000 ký tự (~100 MB)


## 8. Cách 2 — Mô phỏng từng khối `s1`, không xây chuỗi

**Cách cũ vướng ở đâu?** Ta tốn 100 MB chỉ để duyệt `s1 * n1` từ trái sang phải. Nhưng chuỗi đó chỉ là `s1` lặp lại! Duyệt `s1 * n1` **giống hệt** chạy `n1` lần vòng `for ch in s1`.

**Mấu chốt cải tiến:** viết hai vòng lặp lồng nhau:

```python
for block in range(n1):      # khối thứ block
    for ch in s1:            # duyệt từng ký tự của khối
        ... cập nhật j, count như cũ ...
```

Điều **bắt buộc**: `j` và `count` **không được reset** ở đầu mỗi khối. Một bản `s2` hoàn toàn có thể **bắt đầu ở khối này và kết thúc ở khối sau**.

Ví dụ: `s1 = "ba"`, `s2 = "ab"`, `n1 = 2` → `str1 = "baba"`. Trong từng khối `"ba"` riêng lẻ không có `"ab"` nào, nhưng `"ba|ba"` có `a` (khối 1) + `b` (khối 2) = một bản `"ab"` **vắt qua ranh giới khối**.

In [10]:
text = "ba" * 2
print("str1 =", text)
print("Đếm trong từng khối riêng rồi cộng:", count_copies("ba", "ab") + count_copies("ba", "ab"))
print("Đếm trên cả chuỗi (đúng)          :", count_copies(text, "ab"))
# bản "ab" duy nhất dùng chỉ số 1 ('a' của khối 1) và 2 ('b' của khối 2)

str1 = baba
Đếm trong từng khối riêng rồi cộng: 0
Đếm trên cả chuỗi (đúng)          : 1


In [11]:
def repetitions_simulate(s1: str, n1: int, s2: str, n2: int) -> int:
    j = 0            # vị trí đang chờ khớp trong s2 (giữ nguyên qua ranh giới khối!)
    count = 0        # số bản s2 đã ghép xong (K)
    for _ in range(n1):
        for ch in s1:
            if ch == s2[j]:
                j += 1
                if j == len(s2):
                    count += 1
                    j = 0
    return count // n2


for args in [("acb", 4, "ab", 2), ("acb", 1, "acb", 1), ("ba", 2, "ab", 1), ("abab", 9, "bbaa", 2)]:
    print(args, "-> simulate:", repetitions_simulate(*args), "| brute:", repetitions_bruteforce(*args))

('acb', 4, 'ab', 2) -> simulate: 2 | brute: 2
('acb', 1, 'acb', 1) -> simulate: 1 | brute: 1
('ba', 2, 'ab', 1) -> simulate: 1 | brute: 1
('abab', 9, 'bbaa', 2) -> simulate: 2 | brute: 2


### Trace Cách 2 theo từng khối: `s1 = "abab"`, `s2 = "bbaa"`, `n1 = 9`

Ví dụ này "khó" hơn Ví dụ 1 vì mỗi bản `"bbaa"` cần hơn một khối `"abab"`, nên ranh giới bản `s2` rơi lung tung giữa các khối.

Quy ước in: chữ **IN HOA** = ký tự được khớp (dùng), chữ thường = bị xóa, dấu `*` = ngay tại đó vừa ghép xong một bản `s2`. Cột `j` là vị trí đang chờ trong `s2` **ở ranh giới cuối khối** (sau khi đi hết khối).

| Khối | Khớp trong khối | `j` trước → sau | `count` sau khối |
|---:|---|:---:|---:|
| 1 | `a B a B` | 0 → 2 | 0 |
| 2 | `A b A* B` | 2 → 1 | 1 |
| 3 | `a B A b` | 1 → 3 | 1 |
| 4 | `A* B a B` | 3 → 2 | 2 |
| 5 | `A b A* B` | 2 → 1 | 3 |
| 6 | `a B A b` | 1 → 3 | 3 |
| 7 | `A* B a B` | 3 → 2 | 4 |
| 8 | `A b A* B` | 2 → 1 | 5 |
| 9 | `a B A b` | 1 → 3 | 5 |

`K = 5`, `n2 = 2` → đáp án `5 // 2 = 2`. **Bạn có thấy gì lặp lại không?** Hãy nhìn cột `j`: `2, 1, 3, 2, 1, 3, 2, 1, 3`...

In [12]:
def trace_simulate(s1, n1, s2):
    j = count = 0
    for block in range(1, n1 + 1):
        j_before, marks = j, []
        for ch in s1:
            if ch == s2[j]:
                j += 1
                if j == len(s2):
                    count += 1
                    j = 0
                    marks.append(ch.upper() + "*")
                else:
                    marks.append(ch.upper())
            else:
                marks.append(ch)
        print(f"khối {block}: {' '.join(marks):<14} j {j_before} -> {j} | count = {count}")
    return count


K = trace_simulate("abab", 9, "bbaa")
print("K =", K, "-> đáp án với n2 = 2:", K // 2)

khối 1: a B a B        j 0 -> 2 | count = 0
khối 2: A b A* B       j 2 -> 1 | count = 1
khối 3: a B A b        j 1 -> 3 | count = 1
khối 4: A* B a B       j 3 -> 2 | count = 2
khối 5: A b A* B       j 2 -> 1 | count = 3
khối 6: a B A b        j 1 -> 3 | count = 3
khối 7: A* B a B       j 3 -> 2 | count = 4
khối 8: A b A* B       j 2 -> 1 | count = 5
khối 9: a B A b        j 1 -> 3 | count = 5
K = 5 -> đáp án với n2 = 2: 2


### Nút thắt của Cách 2 — và một quan sát vàng

Cách 2 đã hết tốn bộ nhớ (chỉ còn `O(1)`), nhưng **thời gian vẫn là `O(n1 · L1)`** = `10^8` bước tệ nhất. Vẫn quá chậm.

Quan sát vàng từ bảng trace: ở **ranh giới giữa hai khối**, toàn bộ "trạng thái" của quá trình chỉ là **một con số `j`** (đang chờ ký tự nào của `s2`). Vì sao?

- Khối tiếp theo luôn là **cùng một chuỗi `s1`**.
- Cách khớp trong khối chỉ phụ thuộc vào `j` lúc bắt đầu khối.
- `count` chỉ là **kết quả cộng dồn**, nó không ảnh hưởng gì đến cách khớp tiếp theo.

Mà `j` chỉ nhận giá trị trong `0, 1, ..., L2 - 1` — **tối đa `L2 ≤ 100` giá trị**. Trong khi có tới `10^6` khối! Vậy chắc chắn `j` phải **lặp lại**, và một khi nó lặp lại thì **mọi thứ phía sau lặp lại y hệt**. Ta sẽ không cần mô phỏng `10^6` khối nữa — chỉ cần mô phỏng tới khi thấy lặp, rồi **nhảy cóc**.

Trước khi áp vào bài, ta học kỹ thuật này trên một ví dụ mini.

## 9. Kiến thức nền 3: phát hiện chu kỳ (cycle detection)

### 9.1 Dãy sinh bởi một quy tắc cố định

Xét quy tắc `f(x) = (x * x + 1) % 10` và dãy `x0 = 3, x1 = f(x0), x2 = f(x1), ...` (`%` là phép **chia lấy dư**, nên kết quả luôn trong `0..9`).

Tính tay:

```text
x0 = 3
x1 = (3*3 + 1) % 10 = 10 % 10 = 0
x2 = (0*0 + 1) % 10 = 1
x3 = (1*1 + 1) % 10 = 2
x4 = (2*2 + 1) % 10 = 5
x5 = (5*5 + 1) % 10 = 26 % 10 = 6
x6 = (6*6 + 1) % 10 = 37 % 10 = 7
x7 = (7*7 + 1) % 10 = 50 % 10 = 0   <- 0 đã xuất hiện ở x1!
```

Hai ý then chốt:

1. **Nguyên lý chuồng bồ câu (pigeonhole principle)**: chỉ có 10 giá trị có thể, nên trong 11 phần tử `x0..x10` chắc chắn có hai phần tử bằng nhau.
2. **Tất định (deterministic)**: giá trị sau chỉ phụ thuộc giá trị trước. Vì `x7 = x1` nên `x8 = x2`, `x9 = x3`, ... → dãy lặp mãi đoạn `0, 1, 2, 5, 6, 7`.

Hình dạng dãy giống chữ **ρ (rho)**: một **đoạn đầu (prefix)** `3` không lặp, rồi đến một **vòng (cycle)** dài `6`.

### 9.2 Dùng `dict` để ghi nhớ "lần đầu thấy giá trị này ở bước nào"

Để phát hiện lặp bằng code, ta cần một **dictionary (`dict`)** — bảng tra cứu `key → value`:

- `seen = {}` tạo bảng rỗng.
- `seen[x] = step` ghi lại: giá trị `x` được thấy lần đầu ở bước `step`.
- `x in seen` hỏi: giá trị `x` đã từng thấy chưa? (nhanh, trung bình `O(1)`).
- `seen[x]` đọc lại bước đã lưu.

In [13]:
seen = {}
seen[3] = 0          # thấy 3 ở bước 0
seen[0] = 1          # thấy 0 ở bước 1
print(seen)          # {3: 0, 0: 1}
print(0 in seen)     # True : 0 đã từng thấy
print(seen[0])       # 1    : lần đầu thấy ở bước 1
print(5 in seen)     # False

{3: 0, 0: 1}
True
1
False


In [14]:
def f(x):
    return (x * x + 1) % 10


def find_cycle(x0):
    seen = {x0: 0}         # giá trị -> bước lần đầu thấy
    seq = [x0]             # seq[k] = x_k
    x, k = x0, 0
    while True:
        x, k = f(x), k + 1
        seq.append(x)
        if x in seen:
            return seen[x], k, seq     # (p, k): x_k == x_p
        seen[x] = k


p, k, seq = find_cycle(3)
print("Dãy:", seq)
print(f"x_{k} = x_{p} = {seq[k]}  -> prefix dài {p}, chu kỳ dài L = {k - p}")

Dãy: [3, 0, 1, 2, 5, 6, 7, 0]
x_7 = x_1 = 0  -> prefix dài 1, chu kỳ dài L = 6


### 9.3 Nhảy qua chu kỳ: tính `x_N` với `N` khổng lồ

Đã biết `x_k = x_p` với `L = k - p`. Muốn biết `x_N` (ví dụ `N = 10^9`) mà không đi `10^9` bước:

- Nếu `N < k`: đọc thẳng `seq[N]`.
- Nếu `N ≥ k`: sau bước `p`, dãy lặp với chu kỳ `L`, nên `x_N = x_{p + (N - p) % L}`.

Ví dụ: `p = 1`, `L = 6`, `N = 20` → `(20 - 1) % 6 = 19 % 6 = 1` → `x_20 = x_2 = 1`.

Trong Python, `divmod(a, b)` trả về **cùng lúc** thương và dư: `divmod(19, 6) = (3, 1)` — nghĩa là "đi trọn 3 vòng, dư 1 bước". Ta sẽ dùng đúng cặp này trong bài: **số vòng trọn** để cộng dồn, **số bước dư** để xử lý phần đuôi.

In [15]:
def value_after(x0, N):
    p, k, seq = find_cycle(x0)
    if N < k:
        return seq[N]
    L = k - p
    return seq[p + (N - p) % L]


def value_after_slow(x0, N):
    x = x0
    for _ in range(N):
        x = f(x)
    return x


print("divmod(19, 6) =", divmod(19, 6))
assert all(value_after(3, N) == value_after_slow(3, N) for N in range(200))
print("Khớp với cách đi từng bước cho N = 0..199")
print("x_(10^9) =", value_after(3, 10**9), "(tính tức thì, không đi 10^9 bước)")

divmod(19, 6) = (3, 1)
Khớp với cách đi từng bước cho N = 0..199
x_(10^9) = 5 (tính tức thì, không đi 10^9 bước)


## 10. Cách 3 — Phát hiện chu kỳ trên `j` (cách chính)

Ghép mục 8 và mục 9:

| Ví dụ mini (mục 9) | Bài toán của ta |
|---|---|
| bước `k` | số khối `s1` đã xử lý xong |
| giá trị `x_k` | `j` ở ranh giới sau `k` khối |
| quy tắc `f` | "đi hết một khối `s1`" |
| tối đa 10 giá trị | tối đa `L2` giá trị của `j` |

Khác biệt duy nhất: ta không cần `x_N` mà cần **`count` sau `n1` khối**. Nên ngoài `seen`, ta lưu thêm **lịch sử đếm**:

- `seen[j] = k`: lần đầu thấy giá trị `j` ở ranh giới là sau `k` khối.
- `counts[k]`: giá trị `count` sau `k` khối. Bắt đầu `seen = {0: 0}`, `counts = [0]` (trước khối nào, `j = 0`, `count = 0`).

Khi đi xong khối `k` mà `j` đã có trong `seen`, đặt `p = seen[j]`:

- **Độ dài chu kỳ** (tính bằng số khối): `L = k - p`.
- **Mỗi chu kỳ ghép thêm**: `d = counts[k] - counts[p]` bản `s2`.
- **Số khối còn lại**: `rem = n1 - k`. Tách `full, r = divmod(rem, L)`: `full` chu kỳ trọn + `r` khối lẻ.
- `full` chu kỳ trọn cho thêm `full * d` bản.
- `r` khối lẻ bắt đầu từ trạng thái `j` — **giống hệt** trạng thái sau `p` khối — nên chúng cho thêm đúng `counts[p + r] - counts[p]` bản (đã có sẵn trong lịch sử vì `r < L` nên `p + r < k`).

```text
K = counts[k] + full * d + (counts[p + r] - counts[p])
đáp án = K // n2
```

Nếu đi hết `n1` khối mà chưa thấy lặp (xảy ra khi `n1` nhỏ), thì `count` lúc đó chính là `K`.

### Vì sao nhiều nhất chỉ phải mô phỏng `L2` khối?

Các ranh giới sau `0, 1, 2, ..., L2` khối cho ta `L2 + 1` giá trị `j`, mà `j` chỉ có `L2` giá trị khả dĩ (`0..L2-1`). Theo nguyên lý chuồng bồ câu, **chậm nhất sau khối thứ `L2`** đã có một giá trị lặp. Với `L2 ≤ 100`, ta mô phỏng tối đa `100` khối × `100` ký tự = `10^4` bước, thay vì `10^8`. Nhanh hơn **10.000 lần**.

Một trường hợp đặc biệt cần nói rõ: nếu `s2` chứa một ký tự **không có trong `s1`**, con trỏ `j` sẽ kẹt mãi ở ký tự đó, không bao giờ ghép xong bản nào → `K = 0`, đáp án `0`. Thuật toán chu kỳ vẫn tự xử lý đúng (j lặp với `d = 0`), nhưng kiểm tra sớm bằng `set` giúp code rõ ý và trả lời ngay.

In [16]:
def repetitions_cycle(s1: str, n1: int, s2: str, n2: int) -> int:
    if not set(s2) <= set(s1):          # s2 có ký tự không xuất hiện trong s1
        return 0
    L2 = len(s2)
    j = 0
    count = 0
    seen = {0: 0}                       # j ở ranh giới -> số khối k lần đầu thấy
    counts = [0]                        # counts[k] = count sau k khối
    for k in range(1, n1 + 1):
        for ch in s1:                   # đi hết khối thứ k
            if ch == s2[j]:
                j += 1
                if j == L2:
                    count += 1
                    j = 0
        counts.append(count)
        if j in seen:                   # trạng thái lặp lại -> có chu kỳ
            p = seen[j]
            L = k - p                   # độ dài chu kỳ (số khối)
            d = count - counts[p]       # số bản s2 ghép thêm mỗi chu kỳ
            full, r = divmod(n1 - k, L)
            K = count + full * d + (counts[p + r] - counts[p])
            return K // n2
        seen[j] = k
    return count // n2                  # n1 nhỏ: chưa kịp lặp đã hết khối


print(repetitions_cycle("acb", 4, "ab", 2))      # 2  (Ví dụ 1)
print(repetitions_cycle("acb", 1, "acb", 1))     # 1  (Ví dụ 2)
print(repetitions_cycle("abab", 9, "bbaa", 2))   # 2  (khớp trace mục 8)
print(repetitions_cycle("abc", 10**6, "d", 1))   # 0  ('d' không có trong s1)

2
1
2
0


### Trace Cách 3 trên Ví dụ 1: `s1 = "acb"`, `n1 = 4`, `s2 = "ab"`, `n2 = 2`

**Trạng thái ban đầu** (sau 0 khối): `j = 0`, `count = 0`, `seen = {0: 0}`, `counts = [0]`.

**Khối 1** — duyệt từng ký tự của `"acb"`:

| Ký tự | `s2[j]` đang chờ | Khớp? | `j` sau | `count` |
|:---:|:---:|:---:|:---:|---:|
| `a` | `a` (j=0) | có | 1 | 0 |
| `c` | `b` (j=1) | không | 1 | 0 |
| `b` | `b` (j=1) | có → `j = 2 = L2`, xong 1 bản, `j = 0` | 0 | 1 |

Sau khối 1: `j = 0`, `counts = [0, 1]`. Hỏi: `0 in seen`? **Có**, `seen[0] = 0` → `p = 0`, `k = 1`.

**Nhảy qua chu kỳ:**

| Đại lượng | Công thức | Giá trị |
|---|---|---|
| `L` | `k - p = 1 - 0` | `1` khối |
| `d` | `counts[1] - counts[0] = 1 - 0` | `1` bản / chu kỳ |
| `rem` | `n1 - k = 4 - 1` | `3` khối |
| `full, r` | `divmod(3, 1)` | `(3, 0)` |
| `K` | `1 + 3·1 + (counts[0] - counts[0])` | `4` |
| đáp án | `K // n2 = 4 // 2` | **`2`** |

Ý nghĩa bằng lời: mỗi khối `"acb"` đưa `j` từ `0` về lại `0` và ghép đúng 1 bản `"ab"`. Có 4 khối → 4 bản → `4 // 2 = 2` lần `str2`. Khớp đáp án đề.

In [17]:
def trace_cycle(s1, n1, s2, n2, show_chars_block=0):
    L2 = len(s2)
    j = count = 0
    seen, counts = {0: 0}, [0]
    print(f"bắt đầu : j = 0, count = 0, seen = {seen}")
    for k in range(1, n1 + 1):
        for ch in s1:
            need = s2[j]
            if ch == need:
                j += 1
                if j == L2:
                    count += 1
                    j = 0
            if k == show_chars_block:
                print(f"   ký tự {ch!r} chờ {need!r}: {'khớp' if ch == need else 'xóa '} -> j = {j}, count = {count}")
        counts.append(count)
        print(f"sau khối {k}: j = {j}, count = {count}, counts = {counts}")
        if j in seen:
            p = seen[j]
            L, d = k - p, count - counts[p]
            full, r = divmod(n1 - k, L)
            tail = counts[p + r] - counts[p]
            K = count + full * d + tail
            print(f"  j = {j} đã thấy sau {p} khối -> p = {p}, k = {k}, L = {L}, d = {d}")
            print(f"  còn {n1 - k} khối: full = {full} chu kỳ, r = {r} khối lẻ (cho thêm counts[{p + r}] - counts[{p}] = {tail})")
            print(f"  K = {count} + {full}*{d} + {tail} = {K}  -> đáp án = {K} // {n2} = {K // n2}")
            return K // n2
        seen[j] = k
        print(f"  ghi seen[{j}] = {k} -> seen = {seen}")
    print("hết khối, chưa lặp -> đáp án =", count // n2)
    return count // n2


_ = trace_cycle("acb", 4, "ab", 2, show_chars_block=1)

bắt đầu : j = 0, count = 0, seen = {0: 0}
   ký tự 'a' chờ 'a': khớp -> j = 1, count = 0
   ký tự 'c' chờ 'b': xóa  -> j = 1, count = 0
   ký tự 'b' chờ 'b': khớp -> j = 0, count = 1
sau khối 1: j = 0, count = 1, counts = [0, 1]
  j = 0 đã thấy sau 0 khối -> p = 0, k = 1, L = 1, d = 1
  còn 3 khối: full = 3 chu kỳ, r = 0 khối lẻ (cho thêm counts[0] - counts[0] = 0)
  K = 1 + 3*1 + 0 = 4  -> đáp án = 4 // 2 = 2


### Trace Cách 3 trên ví dụ "khó": `s1 = "abab"`, `n1 = 9`, `s2 = "bbaa"`, `n2 = 2`

Ví dụ 1 hơi "hiền" (chu kỳ dài 1 khối, bắt đầu ngay từ đầu, không có khối lẻ). Ví dụ này có đủ ba thứ: **đoạn đầu (prefix) không lặp**, **chu kỳ dài nhiều khối**, và **khối lẻ có đóng góp**. Dùng lại bảng mục 8:

| Sau `k` khối | `j` | `count` | Hành động |
|---:|---:|---:|---|
| 0 | 0 | 0 | `seen = {0: 0}` |
| 1 | 2 | 0 | chưa thấy 2 → `seen[2] = 1` |
| 2 | 1 | 1 | chưa thấy 1 → `seen[1] = 2` |
| 3 | 3 | 1 | chưa thấy 3 → `seen[3] = 3` |
| 4 | 2 | 2 | **2 đã thấy sau 1 khối** → `p = 1`, `k = 4` |

- `L = 4 - 1 = 3` khối, `d = counts[4] - counts[1] = 2 - 0 = 2` bản mỗi chu kỳ.
- Còn `9 - 4 = 5` khối: `divmod(5, 3) = (1, 2)` → 1 chu kỳ trọn + 2 khối lẻ.
- 2 khối lẻ bắt đầu từ trạng thái giống "sau 1 khối", nên cho thêm `counts[1 + 2] - counts[1] = counts[3] - counts[1] = 1 - 0 = 1` bản.
- `K = 2 + 1·2 + 1 = 5` (khớp `K = 5` của bảng mô phỏng đủ 9 khối ở mục 8!) → đáp án `5 // 2 = 2`.

Ở đây khối 1 là prefix: trạng thái `j = 0` ban đầu không bao giờ quay lại. Đó là lý do ta phải nhớ `p` chứ không mặc định chu kỳ bắt đầu từ khối 0.

In [18]:
_ = trace_cycle("abab", 9, "bbaa", 2)
print("Đối chiếu brute force:", repetitions_bruteforce("abab", 9, "bbaa", 2))

bắt đầu : j = 0, count = 0, seen = {0: 0}
sau khối 1: j = 2, count = 0, counts = [0, 0]
  ghi seen[2] = 1 -> seen = {0: 0, 2: 1}
sau khối 2: j = 1, count = 1, counts = [0, 0, 1]
  ghi seen[1] = 2 -> seen = {0: 0, 2: 1, 1: 2}
sau khối 3: j = 3, count = 1, counts = [0, 0, 1, 1]
  ghi seen[3] = 3 -> seen = {0: 0, 2: 1, 1: 2, 3: 3}
sau khối 4: j = 2, count = 2, counts = [0, 0, 1, 1, 2]
  j = 2 đã thấy sau 1 khối -> p = 1, k = 4, L = 3, d = 2
  còn 5 khối: full = 1 chu kỳ, r = 2 khối lẻ (cho thêm counts[3] - counts[1] = 1)
  K = 2 + 1*2 + 1 = 5  -> đáp án = 5 // 2 = 2
Đối chiếu brute force: 2


### Trường hợp `s2` có ký tự không có trong `s1`

`set(s)` tạo **tập hợp (set)** các ký tự khác nhau của `s`; `A <= B` với hai set nghĩa là "`A` là tập con của `B`" (mọi phần tử của `A` đều nằm trong `B`).

- `set("ab") <= set("acb")` → `True`: mọi ký tự của `s2` đều có trong `s1`, có thể ghép được.
- `set("ad") <= set("acb")` → `False`: chữ `d` không bao giờ khớp, `j` kẹt ở chỗ chờ `d` mãi mãi → đáp án `0`.

Cell dưới cho thấy cả bản có kiểm tra sớm lẫn bản mô phỏng (không kiểm tra) đều trả `0`, nhưng với `n1 = 10^6` thì kiểm tra sớm trả lời ngay lập tức.

In [19]:
print(sorted(set("ab")), "<=", sorted(set("acb")), "->", set("ab") <= set("acb"))   # sorted chỉ để in gọn, có thứ tự
print(sorted(set("ad")), "<=", sorted(set("acb")), "->", set("ad") <= set("acb"))

print(repetitions_cycle("acb", 10**6, "ad", 1))        # 0, trả lời ngay
print(repetitions_simulate("acb", 1000, "ad", 1))      # 0, nhưng phải đi hết 1000 khối
_ = trace_cycle("acb", 10**6, "ad", 1)                 # không kiểm tra set: vẫn ra 0 vì d = 0

['a', 'b'] <= ['a', 'b', 'c'] -> True
['a', 'd'] <= ['a', 'b', 'c'] -> False
0
0
bắt đầu : j = 0, count = 0, seen = {0: 0}
sau khối 1: j = 1, count = 0, counts = [0, 0]
  ghi seen[1] = 1 -> seen = {0: 0, 1: 1}
sau khối 2: j = 1, count = 0, counts = [0, 0, 0]
  j = 1 đã thấy sau 1 khối -> p = 1, k = 2, L = 1, d = 0
  còn 999998 khối: full = 999998 chu kỳ, r = 0 khối lẻ (cho thêm counts[1] - counts[1] = 0)
  K = 0 + 999998*0 + 0 = 0  -> đáp án = 0 // 1 = 0


In [20]:
import random

random.seed(466)
for _ in range(3000):
    a = "".join(random.choice("abc") for _ in range(random.randint(1, 6)))
    b = "".join(random.choice("abc") for _ in range(random.randint(1, 5)))
    n1 = random.randint(1, 30)
    n2 = random.randint(1, 5)
    expected = repetitions_bruteforce(a, n1, b, n2)
    assert repetitions_simulate(a, n1, b, n2) == expected, (a, n1, b, n2)
    assert repetitions_cycle(a, n1, b, n2) == expected, (a, n1, b, n2)

# thêm: đối chiếu với định nghĩa nguyên văn của đề trên input rất nhỏ
for _ in range(300):
    a = "".join(random.choice("ab") for _ in range(random.randint(1, 4)))
    b = "".join(random.choice("ab") for _ in range(random.randint(1, 3)))
    n1, n2 = random.randint(1, 6), random.randint(1, 3)
    assert max_m_by_definition(a, n1, b, n2) == repetitions_cycle(a, n1, b, n2), (a, n1, b, n2)
print("3000 + 300 test ngẫu nhiên: brute == simulate == cycle == định nghĩa đề")

3000 + 300 test ngẫu nhiên: brute == simulate == cycle == định nghĩa đề


## 11. Cách 4 — Bảng tiền xử lý `next_j / gain` (tùy chọn, rất đáng biết)

Quan sát ở mục 8 nói rằng: "đi hết một khối `s1`" là một **hàm** biến `j` lúc đầu khối thành `j` lúc cuối khối, kèm theo số bản `s2` ghép được trong khối đó. Hàm này **chỉ có `L2` đầu vào** (`j = 0..L2-1`), nên ta có thể **tính sẵn toàn bộ** thành hai mảng:

- `next_j[start]`: nếu đầu khối đang chờ `s2[start]`, thì cuối khối đang chờ `s2[next_j[start]]`.
- `gain[start]`: số bản `s2` ghép xong **trong** khối đó.

Chi phí lập bảng: `L2` lần duyệt `s1` → `O(L1 · L2) = 10^4`.

Sau khi có bảng, mỗi khối chỉ còn **2 phép tra mảng**:

```python
count += gain[j]
j = next_j[j]
```

thay vì duyệt `L1` ký tự. Đây là ý tưởng "**nén một khối thành một bước**".

In [21]:
def build_table(s1: str, s2: str):
    L2 = len(s2)
    next_j = [0] * L2
    gain = [0] * L2
    for start in range(L2):
        j, c = start, 0
        for ch in s1:
            if ch == s2[j]:
                j += 1
                if j == L2:
                    c += 1
                    j = 0
        next_j[start], gain[start] = j, c
    return next_j, gain


next_j, gain = build_table("abab", "bbaa")
print("start :", list(range(4)))
print("next_j:", next_j)
print("gain  :", gain)
# Đọc: bắt đầu khối khi chờ s2[2] -> cuối khối chờ s2[1], ghép thêm 1 bản (khớp khối 2 của trace mục 8)

start : [0, 1, 2, 3]
next_j: [2, 3, 1, 2]
gain  : [0, 0, 1, 1]


### Đọc bảng `"abab"` / `"bbaa"` như một đồ thị

Từ bảng vừa in, đi từ `j = 0`:

```text
0 --(+0)--> 2 --(+1)--> 1 --(+0)--> 3 --(+1)--> 2 --(+1)--> 1 --> 3 --> 2 ...
```

Bạn thấy lại đúng hình chữ ρ: `0` là đoạn đầu, vòng `2 → 1 → 3 → 2` có tổng gain `1 + 0 + 1 = 2` (= `d` ở mục 10). Vậy bảng `next_j` chính là quy tắc `f` của mục 9, chỉ là "kèm theo điểm thưởng `gain`".

Với bảng này có hai cách đi `n1` khối:

1. **Đi thẳng** `n1` bước tra bảng: `O(L1·L2 + n1)` — với `n1 = 10^6` vẫn chấp nhận được (khoảng một triệu phép tra mảng).
2. **Đi + nhảy chu kỳ** trên bảng: giống Cách 3 nhưng mỗi khối chỉ tốn `O(1)` → tổng `O(L1·L2)`.

Code dưới viết cả hai (bản 2 là để bạn thấy cùng một khung chu kỳ dùng lại y nguyên).

In [22]:
def repetitions_table_walk(s1: str, n1: int, s2: str, n2: int) -> int:
    next_j, gain = build_table(s1, s2)
    j = count = 0
    for _ in range(n1):
        count += gain[j]
        j = next_j[j]
    return count // n2


def repetitions_table_cycle(s1: str, n1: int, s2: str, n2: int) -> int:
    next_j, gain = build_table(s1, s2)
    j = count = 0
    seen, counts = {0: 0}, [0]
    for k in range(1, n1 + 1):
        count += gain[j]
        j = next_j[j]
        counts.append(count)
        if j in seen:
            p = seen[j]
            full, r = divmod(n1 - k, k - p)
            return (count + full * (count - counts[p]) + counts[p + r] - counts[p]) // n2
        seen[j] = k
    return count // n2


for args in [("acb", 4, "ab", 2), ("acb", 1, "acb", 1), ("abab", 9, "bbaa", 2), ("acb", 7, "ad", 1)]:
    print(args, "-> walk:", repetitions_table_walk(*args), "| table+cycle:", repetitions_table_cycle(*args))

('acb', 4, 'ab', 2) -> walk: 2 | table+cycle: 2
('acb', 1, 'acb', 1) -> walk: 1 | table+cycle: 1
('abab', 9, 'bbaa', 2) -> walk: 2 | table+cycle: 2
('acb', 7, 'ad', 1) -> walk: 0 | table+cycle: 0


In [23]:
random.seed(4661)
for _ in range(3000):
    a = "".join(random.choice("abc") for _ in range(random.randint(1, 6)))
    b = "".join(random.choice("abc") for _ in range(random.randint(1, 5)))
    n1, n2 = random.randint(1, 30), random.randint(1, 5)
    expected = repetitions_bruteforce(a, n1, b, n2)
    assert repetitions_table_walk(a, n1, b, n2) == expected, (a, n1, b, n2)
    assert repetitions_table_cycle(a, n1, b, n2) == expected, (a, n1, b, n2)
print("3000 test ngẫu nhiên: table_walk == table_cycle == brute force")

3000 test ngẫu nhiên: table_walk == table_cycle == brute force


## 12. Code nộp LeetCode

Ta nộp **Cách 3 (phát hiện chu kỳ)**: ngắn, không cần bảng phụ, thời gian `O(L1 · L2)` không phụ thuộc `n1`.

Khi chuyển sang `class Solution`:

- Giữ đúng tên method `getMaxRepetitions` và thứ tự tham số `(s1, n1, s2, n2)`.
- Bỏ hết `print`.
- Đặt tên biến rõ nghĩa: `index` (= `j`), `count`, `seen`, `history` (= `counts`).

In [24]:
class Solution:
    def getMaxRepetitions(self, s1: str, n1: int, s2: str, n2: int) -> int:
        if not set(s2) <= set(s1):
            return 0
        m = len(s2)
        index = 0                    # vị trí đang chờ khớp trong s2
        count = 0                    # số bản s2 đã ghép xong
        seen = {0: 0}                # index ở ranh giới -> số khối lần đầu thấy
        history = [0]                # history[k] = count sau k khối
        for k in range(1, n1 + 1):
            for ch in s1:
                if ch == s2[index]:
                    index += 1
                    if index == m:
                        count += 1
                        index = 0
            history.append(count)
            if index in seen:
                p = seen[index]
                full, r = divmod(n1 - k, k - p)
                total = count + full * (count - history[p]) + (history[p + r] - history[p])
                return total // n2
            seen[index] = k
        return count // n2

In [25]:
sol = Solution()

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<38} PASS ({actual})")

check("Ví dụ 1", sol.getMaxRepetitions("acb", 4, "ab", 2), 2)
check("Ví dụ 2", sol.getMaxRepetitions("acb", 1, "acb", 1), 1)
check("s2 có ký tự lạ", sol.getMaxRepetitions("abc", 5, "d", 1), 0)
check("K < n2 nên đáp án 0", sol.getMaxRepetitions("a", 1, "a", 2), 0)
check("bản s2 vắt qua ranh giới khối", sol.getMaxRepetitions("ba", 2, "ab", 1), 1)
check("prefix + chu kỳ 3 khối + khối lẻ", sol.getMaxRepetitions("abab", 9, "bbaa", 2), 2)
check("s2 dài hơn cả s1", sol.getMaxRepetitions("ab", 7, "ababa", 1), 2)
check("toàn chữ a: 9 chữ a -> 4 bản 'aa'", sol.getMaxRepetitions("aaa", 3, "aa", 1), 4)
check("n1 = n2 = 10^6, s1 = s2 = 'a'", sol.getMaxRepetitions("a", 10**6, "a", 10**6), 1)
check("n1 = 10^6, s2 = 'cba'", sol.getMaxRepetitions("abc", 10**6, "cba", 1),
      repetitions_table_walk("abc", 10**6, "cba", 1))

Ví dụ 1                                PASS (2)
Ví dụ 2                                PASS (1)
s2 có ký tự lạ                         PASS (0)
K < n2 nên đáp án 0                    PASS (0)
bản s2 vắt qua ranh giới khối          PASS (1)
prefix + chu kỳ 3 khối + khối lẻ       PASS (2)
s2 dài hơn cả s1                       PASS (2)
toàn chữ a: 9 chữ a -> 4 bản 'aa'      PASS (4)
n1 = n2 = 10^6, s1 = s2 = 'a'          PASS (1)


n1 = 10^6, s2 = 'cba'                  PASS (499999)


In [26]:
random.seed(2024)
for _ in range(2000):
    a = "".join(random.choice("abcd") for _ in range(random.randint(1, 7)))
    b = "".join(random.choice("abcd") for _ in range(random.randint(1, 6)))
    n1, n2 = random.randint(1, 40), random.randint(1, 6)
    assert sol.getMaxRepetitions(a, n1, b, n2) == repetitions_bruteforce(a, n1, b, n2), (a, n1, b, n2)
print("2000 test ngẫu nhiên: Solution == brute force")

# Ca lớn nhất theo constraints: L1 = L2 = 100, n1 = n2 = 10^6
random.seed(7)
worst = []
for _ in range(5):
    a = "".join(random.choice("ab") for _ in range(100))
    b = "".join(random.choice("ab") for _ in range(100))
    t0 = time.perf_counter()
    ans = sol.getMaxRepetitions(a, 10**6, b, 1)
    worst.append(time.perf_counter() - t0)
    assert ans == repetitions_table_walk(a, 10**6, b, 1)
print(f"5 ca L1 = L2 = 100, n1 = 10^6: chậm nhất {max(worst) * 1000:.1f} ms, khớp table_walk")

2000 test ngẫu nhiên: Solution == brute force


5 ca L1 = L2 = 100, n1 = 10^6: chậm nhất 0.0 ms, khớp table_walk


## 13. Vì sao thuật toán đúng?

Lập luận gồm 4 mắt xích, mỗi mắt xích đã được kiểm chứng ở một mục:

1. **Quy về `K`** (mục 2, 6): `[str2, m] = [s2, n2·m]`, và tính đơn điệu cho ta `m_max = K // n2`, với `K` = số bản `s2` ghép liền tối đa là subsequence của `str1`.
2. **Tham lam cho `K` tối đa** (mục 4, 5): khớp mỗi ký tự càng sớm càng tốt không bao giờ làm mất cơ hội cho các ký tự sau, nên con trỏ `j` quay vòng đếm đúng `K`.
3. **Trạng thái ở ranh giới khối chỉ là `j`** (mục 8): mọi khối giống hệt nhau, nên cách khớp của khối tiếp theo chỉ phụ thuộc `j` lúc vào khối. Đây là **bất biến (invariant)** then chốt: *"sau `k` khối, `j` và `count` đúng bằng những gì brute force có sau `k · L1` ký tự"*.
4. **Nhảy chu kỳ** (mục 9, 10): nếu `j` sau `k` khối bằng `j` sau `p` khối thì các khối `k+1, k+2, ...` lặp lại đúng diễn biến của các khối `p+1, p+2, ...`. Do đó cứ mỗi `L = k - p` khối, `count` tăng đúng `d`; và `r < L` khối lẻ tăng đúng `counts[p + r] - counts[p]`.

Không bỏ sót (mắt xích 2: tham lam tối đa) và không đếm lố (mắt xích 3–4: nhảy cóc chỉ là "chép lại" diễn biến có thật). Ngoài ra ta đã đối chiếu với **định nghĩa nguyên văn** của đề trên hàng trăm input ngẫu nhiên.

## 14. Độ phức tạp và so sánh các cách

Gọi `L1 = len(s1) ≤ 100`, `L2 = len(s2) ≤ 100`, `n1 ≤ 10^6`.

| Cách | Ý tưởng | Time | Space | Với constraints tệ nhất |
|---|---|---|---|---|
| 1. Brute force | xây `s1 * n1` rồi đếm | `O(n1 · L1)` | `O(n1 · L1)` | `10^8` bước, ~100 MB → **TLE/MLE** |
| 2. Mô phỏng khối | 2 vòng lồng, giữ `j, count` | `O(n1 · L1)` | `O(1)` | `10^8` bước → **TLE** |
| 3. Chu kỳ (nộp) | dừng khi `j` lặp, nhảy cóc | `O(L1 · L2)` | `O(L2)` | `≤ 10^4` bước, tức thì |
| 4a. Bảng + đi thẳng | lập `next_j/gain`, đi `n1` bước | `O(L1 · L2 + n1)` | `O(L2)` | `~10^6` phép tra, chấp nhận được |
| 4b. Bảng + chu kỳ | như 3 nhưng mỗi khối `O(1)` | `O(L1 · L2)` | `O(L2)` | tức thì |

Giải thích chữ `O(L1 · L2)` của Cách 3: tối đa `L2` khối được mô phỏng (nguyên lý chuồng bồ câu, mục 10), mỗi khối duyệt `L1` ký tự. `seen` và `counts` giữ tối đa `L2 + 1` phần tử → `O(L2)` bộ nhớ. Kiểm tra `set` thêm `O(L1 + L2)`, không đổi bậc.

**Nên chọn cách nào?** Phỏng vấn/nộp bài: Cách 3. Khi bài có **nhiều truy vấn** với cùng `s1, s2` nhưng `n1` khác nhau, hoặc cần phát triển tiếp (nhảy nhị phân — bài tập 6): Cách 4 linh hoạt hơn vì bảng chỉ lập một lần.

## 15. Lỗi thường gặp

### Lỗi 1 — Reset `j` ở đầu mỗi khối (đếm trong từng khối rồi nhân `n1`)

```python
# SAI
per_block = count_copies(s1, s2)     # đếm trong một khối riêng lẻ
return per_block * n1 // n2
```

**Vì sao sai:** bỏ mất các bản `s2` **vắt qua ranh giới khối**, và cả những bản `s2` dài hơn `s1` (cần nhiều khối mới ghép xong một bản). Ví dụ `s1 = "ba"`, `s2 = "ab"`: mỗi khối riêng cho 0, nhưng `"baba"` có 1 bản. Hay `s1 = "ab"`, `s2 = "ababa"` (`s2` dài hơn `s1`): bản sai luôn ra 0.

**Sửa:** `j` phải sống xuyên suốt qua các khối (Cách 2, 3).

In [27]:
def wrong_reset_each_block(s1, n1, s2, n2):
    return count_copies(s1, s2) * n1 // n2


for args in [("ba", 2, "ab", 1), ("ab", 7, "ababa", 1), ("acb", 4, "ab", 2)]:
    print(args, "-> sai:", wrong_reset_each_block(*args), "| đúng:", repetitions_cycle(*args))
# Ví dụ 1 tình cờ đúng vì mỗi khối "acb" tự chứa trọn một "ab" -> test mỗi ví dụ đề là chưa đủ!

('ba', 2, 'ab', 1) -> sai: 0 | đúng: 1
('ab', 7, 'ababa', 1) -> sai: 0 | đúng: 2
('acb', 4, 'ab', 2) -> sai: 2 | đúng: 2


### Lỗi 2 — Quên khối lẻ (hoặc lệch một khối) khi nhảy chu kỳ

```python
# SAI: chỉ cộng các chu kỳ trọn, bỏ r khối lẻ
full = (n1 - k) // L
K = count + full * d
```

**Vì sao sai:** `n1 - k` không phải lúc nào cũng chia hết cho `L`. Ở ví dụ `"abab", 9, "bbaa"`: còn 5 khối, `L = 3` → 1 chu kỳ trọn **+ 2 khối lẻ** cho thêm 1 bản. Bỏ phần lẻ ra `K = 4` thay vì `5`.

Một biến thể của lỗi này: tính `rem = n1 - k - 1` hoặc `n1 - p` do nhầm "sau khối thứ `k`" với "trước khối thứ `k`". Cách phòng: luôn định nghĩa rõ **`counts[k]` = count SAU `k` khối**, và `seen = {0: 0}` nghĩa là "sau 0 khối, `j = 0`".

**Sửa:** `full, r = divmod(n1 - k, L)` và cộng thêm `counts[p + r] - counts[p]`.

In [28]:
def wrong_no_remainder(s1, n1, s2, n2):
    L2, j, count = len(s2), 0, 0
    seen, counts = {0: 0}, [0]
    for k in range(1, n1 + 1):
        for ch in s1:
            if ch == s2[j]:
                j += 1
                if j == L2:
                    count, j = count + 1, 0
        counts.append(count)
        if j in seen:
            p = seen[j]
            full = (n1 - k) // (k - p)
            return (count + full * (count - counts[p])) // n2   # quên r khối lẻ
        seen[j] = k
    return count // n2


print("n2 = 1 để thấy rõ K:")
print("sai :", wrong_no_remainder("abab", 9, "bbaa", 1))   # 4
print("đúng:", repetitions_cycle("abab", 9, "bbaa", 1))    # 5

n2 = 1 để thấy rõ K:
sai : 4
đúng: 5


### Lỗi 3 — Dùng `(j, count)` làm trạng thái

```python
# SAI (chậm): state = (j, count)
if (j, count) in seen: ...
```

**Vì sao sai:** `count` chỉ tăng, không bao giờ quay lại giá trị cũ, nên cặp `(j, count)` gần như không bao giờ lặp → thuật toán "tụt" về Cách 2, mô phỏng đủ `n1` khối, TLE. Trạng thái phải là thứ **quyết định tương lai** — chỉ `j`. `count` là **kết quả** đi kèm, lưu riêng trong `counts`.

### Lỗi 4 — Quên chia `n2`, hoặc chia sai kiểu

- Trả `K` thay vì `K // n2`: Ví dụ 1 sẽ ra `4` thay vì `2`.
- Dùng `K / n2`: ra số thực `2.0`, sai kiểu trả về `int`.
- Dùng `round(K / n2)` hoặc làm tròn lên: `5 / 2 = 2.5` → `3` là sai, phải **làm tròn xuống** vì cần `n2 · m ≤ K`.

### Lỗi 5 — Nhầm "thu được" với substring

`s2 in s1 * n1` hay `(s1 * n1).count(s2)` đếm **chuỗi con liên tiếp**, trong khi đề cho phép **xóa ký tự**. Ví dụ 1: `("acb" * 4).count("ab")` bằng `0`, nhưng đáp án thật cần `K = 4`.

### Lỗi 6 — Xây chuỗi khổng lồ trong bản nộp

`s1 * n1` với `n1 = 10^6`, `L1 = 100` tốn ~100 MB và vòng lặp `10^8` bước → Memory/Time Limit Exceeded. Brute force chỉ để làm trọng tài trên input nhỏ.

In [29]:
# Lỗi 3: đếm số khối phải mô phỏng khi trạng thái là j so với (j, count)
def blocks_until_repeat(s1, n1, s2, use_count_in_state):
    L2, j, count = len(s2), 0, 0
    seen = {(0, 0) if use_count_in_state else 0}
    for k in range(1, n1 + 1):
        for ch in s1:
            if ch == s2[j]:
                j += 1
                if j == L2:
                    count, j = count + 1, 0
        state = (j, count) if use_count_in_state else j
        if state in seen:
            return k
        seen.add(state)
    return n1


print("state = j          : lặp sau", blocks_until_repeat("acb", 10**5, "ab", False), "khối")
print("state = (j, count) : lặp sau", blocks_until_repeat("acb", 10**5, "ab", True), "khối (= n1, không bao giờ lặp)")

# Lỗi 4 và 5
print("Quên // n2      :", count_copies("acb" * 4, "ab"), "(phải là 2)")
print("Nhầm substring  :", ("acb" * 4).count("ab"), "(K thật là", count_copies("acb" * 4, "ab"), ")")

state = j          : lặp sau 1 khối


state = (j, count) : lặp sau 100000 khối (= n1, không bao giờ lặp)
Quên // n2      : 4 (phải là 2)
Nhầm substring  : 0 (K thật là 4 )


## 16. Mẹo nhận dạng pattern

Khi đọc đề thấy các dấu hiệu sau, hãy nghĩ ngay tới **"mô phỏng theo khối + phát hiện chu kỳ"**:

- Một thao tác **lặp lại rất nhiều lần** (`n` tới `10^6`, `10^9`, thậm chí `10^18`), nhưng mỗi lần **giống hệt nhau**.
- Có thể tóm gọn trạng thái giữa hai lần lặp thành **một thứ nhỏ, hữu hạn** (ở đây: `j` chỉ có `≤ 100` giá trị; ở bài khác: một mặt nạ bit 8 ô, một vị trí trên vòng tròn...).
- Đại lượng cần tính là **tổng cộng dồn** (đếm, cộng điểm) dọc quá trình → lưu thêm lịch sử `counts[k]` để nhảy cóc.

Công thức tư duy 3 câu:

1. *"Trạng thái tối thiểu quyết định tương lai là gì?"* → `j`.
2. *"Có bao nhiêu trạng thái?"* → `L2` → chắc chắn lặp sau `≤ L2` bước.
3. *"Khi lặp: cộng `full` chu kỳ trọn + `r` bước lẻ tra trong lịch sử."*

Và với chữ "**thu được bằng cách xóa ký tự**" → nghĩ ngay **subsequence + hai con trỏ tham lam**.

## 17. Biến thể và bài liên quan

| Bài | Liên hệ với bài này |
|---|---|
| LeetCode 392 — Is Subsequence | Đúng hàm `is_subsequence` hai con trỏ ở mục 4. |
| LeetCode 792 — Number of Matching Subsequences | Kiểm tra nhiều `pattern` là subsequence của một `text`; tối ưu bằng gom nhóm theo ký tự đang chờ. |
| LeetCode 1055 — Shortest Way to Form String (Premium) | Ngược lại bài này: cần **ít nhất** bao nhiêu khối `source` để ghép ra `target` (bài tập 5). |
| LeetCode 686 — Repeated String Match | Lặp chuỗi `a` ít nhất bao nhiêu lần để `b` là **substring** (liên tiếp) — so sánh để thấy khác biệt substring/subsequence. |
| LeetCode 957 — Prison Cells After N Days | Cùng kỹ thuật **phát hiện chu kỳ rồi nhảy cóc** với `N` tới `10^9`. |
| LeetCode 202 — Happy Number | Phát hiện chu kỳ của dãy sinh bởi một quy tắc cố định (như mục 9). |

Kỹ thuật **bảng `next_j` + nhảy nhị phân (binary lifting)** (bài tập 6) là cách tổng quát hóa Cách 4: thay vì tìm chu kỳ, ta lập sẵn "đi `2^t` khối thì tới đâu, được bao nhiêu" rồi ghép theo biểu diễn nhị phân của `n1`.

## 18. Bài tập tự luyện

Làm lần lượt, **đừng xem đáp án** ở cuối notebook trước khi tự thử. Các cell TODO chỉ định nghĩa hàm (chưa được gọi), test để ở dạng comment — làm xong hãy bỏ dấu `#` để tự kiểm tra.

### Bài 1 — Đoán output / hiểu định nghĩa (dễ)

Không chạy code, hãy trả lời:

1. `["ab", 3]` là chuỗi gì?
2. `"aab"` có thu được từ `"abab"` không? Nếu có, giữ lại những chỉ số nào (theo cách khớp tham lam)?
3. `"ba"` có thu được từ `"ab"` không?
4. Với `s1 = "ab"`, `n1 = 3`, `s2 = "ba"`, `n2 = 1`: tính `K` và đáp án. Có bản `"ba"` nào vắt qua ranh giới khối không?

### Bài 2 — Trace chu kỳ bằng tay (dễ–vừa)

Với `s1 = "abc"`, `n1 = 10`, `s2 = "cba"`, `n2 = 2`:

1. Lập bảng giống mục 10: sau mỗi khối `k = 0, 1, 2, ...` ghi `j`, `count`, `seen` cho tới khi `j` lặp.
2. Tìm `p`, `k`, `L`, `d`, `full`, `r` và phần đóng góp của khối lẻ.
3. Tính `K` và đáp án.

(Gợi ý nhỏ: mỗi bản `"cba"` cần chữ `c` của một khối, `b` của khối sau, `a` của khối sau nữa.)

### Bài 3 — Điền TODO: mô phỏng từng khối (vừa)

Hoàn thiện `exercise_simulate` (chính là Cách 2) theo các TODO.

In [30]:
def exercise_simulate(s1: str, n1: int, s2: str, n2: int) -> int:
    # TODO 1: khởi tạo j = 0 (vị trí chờ trong s2) và count = 0
    # TODO 2: lặp n1 khối; trong mỗi khối duyệt từng ký tự ch của s1
    # TODO 3: nếu ch == s2[j] thì tăng j; nếu j == len(s2) thì count += 1 và j = 0
    # TODO 4: KHÔNG reset j ở đầu mỗi khối
    # TODO 5: trả về count // n2
    pass

# Làm xong rồi mới bỏ comment để test:
# assert exercise_simulate("acb", 4, "ab", 2) == 2
# assert exercise_simulate("ba", 2, "ab", 1) == 1
# assert exercise_simulate("abab", 9, "bbaa", 2) == 2

### Bài 4 — Tự viết lại Cách 3, nhưng xử lý khối lẻ bằng mô phỏng (vừa–khó)

Viết `exercise_cycle(s1, n1, s2, n2)` phát hiện chu kỳ trên `j`. Khác bản trong notebook: sau khi cộng `full` chu kỳ trọn, **không** tra `counts[p + r]` mà **mô phỏng tiếp đúng `r` khối** từ trạng thái `j` hiện tại để cộng phần lẻ. (Cách này dễ nghĩ hơn, và vẫn nhanh vì `r < L ≤ L2`.)

Yêu cầu: không dùng `s1 * n1`; chỉ mô phỏng tối đa khoảng `2 · L2` khối.

In [31]:
def exercise_cycle(s1: str, n1: int, s2: str, n2: int) -> int:
    # TODO 1: nếu s2 có ký tự không có trong s1 -> return 0
    # TODO 2: j = 0, count = 0, seen = {0: (0, 0)}  # j -> (k, count lúc đó)
    # TODO 3: lặp k = 1..n1, đi hết một khối s1 (viết hàm con run_block(j) -> (j_mới, số bản ghép được) cho gọn)
    # TODO 4: nếu j đã có trong seen: tính L, d, full, r; count += full * d
    #         rồi gọi run_block thêm r lần để cộng phần lẻ; return count // n2
    # TODO 5: nếu hết n1 khối mà chưa lặp: return count // n2
    raise NotImplementedError

# assert exercise_cycle("acb", 4, "ab", 2) == 2
# assert exercise_cycle("abab", 9, "bbaa", 1) == 5
# assert exercise_cycle("abc", 10**6, "d", 1) == 0

### Bài 5 — Biến thể: Shortest Way to Form String (LeetCode 1055, Premium) (vừa)

Một **subsequence** của `source` là chuỗi thu được bằng cách xóa ký tự của `source`. Cho `source` và `target`, hãy trả về **số ít nhất** các subsequence của `source` mà khi nối lại bằng đúng `target`; nếu không thể, trả `-1`.

```text
source = "abc", target = "abcbc"  -> 2   ("abc" + "bc")
source = "abc", target = "acdbc"  -> -1  (không có 'd' trong source)
source = "xyz", target = "xzyxz"  -> 3   ("xz" + "y" + "xz")
```

Liên hệ: đây chính là hỏi "phải đi qua **ít nhất bao nhiêu khối `source`** để khớp tham lam hết **một** bản `target`" — con trỏ chạy trên `target`, các khối là `source`.

In [32]:
def exercise_shortest_way(source: str, target: str) -> int:
    # TODO 1: nếu target có ký tự không có trong source -> return -1
    # TODO 2: i = 0 (vị trí chờ trong target), blocks = 0
    # TODO 3: while i < len(target): blocks += 1, rồi duyệt một lượt source,
    #         gặp ch == target[i] thì i += 1 (nhớ kiểm tra i < len(target))
    # TODO 4: return blocks
    pass

# assert exercise_shortest_way("abc", "abcbc") == 2
# assert exercise_shortest_way("abc", "acdbc") == -1
# assert exercise_shortest_way("xyz", "xzyxz") == 3

### Bài 6 — Mini challenge: nhảy nhị phân (binary lifting) trên bảng `next_j` (khó)

Không dùng phát hiện chu kỳ. Hãy lập bảng "đi `2^t` khối":

- `jump[0][j] = next_j[j]`, `add[0][j] = gain[j]` (bảng của Cách 4).
- `jump[t][j] = jump[t-1][ jump[t-1][j] ]` (đi `2^(t-1)` khối, rồi thêm `2^(t-1)` khối nữa).
- `add[t][j] = add[t-1][j] + add[t-1][ jump[t-1][j] ]`.

Rồi phân tích `n1` theo nhị phân (ví dụ `13 = 8 + 4 + 1`) và ghép các bước nhảy tương ứng, bắt đầu từ `j = 0`.

Viết `exercise_binary_lifting(s1, n1, s2, n2)` và cho biết độ phức tạp.

In [33]:
def exercise_binary_lifting(s1: str, n1: int, s2: str, n2: int) -> int:
    # TODO 1: next_j, gain = build_table(s1, s2)
    # TODO 2: jump = [next_j], add = [gain]; lặp t trong khi 2^t <= n1 để thêm tầng mới
    # TODO 3: j = 0, count = 0; với mỗi bit t của n1 đang bật: count += add[t][j]; j = jump[t][j]
    # TODO 4: return count // n2
    pass

# assert exercise_binary_lifting("acb", 4, "ab", 2) == 2
# assert exercise_binary_lifting("abab", 9, "bbaa", 2) == 2
# assert exercise_binary_lifting("abc", 10**6, "cba", 1) == repetitions_table_walk("abc", 10**6, "cba", 1)

## 19. Gợi ý theo 3 mức

**Bài 1**
- Mức 1: nhớ `[s, n]` là nhân chuỗi; "thu được" là xóa ký tự, giữ thứ tự.
- Mức 2: dùng hai con trỏ khớp tham lam, ghi lại chỉ số được dùng.
- Mức 3: với câu 4, viết `"ababab"` và đánh chỉ số `0..5`; tìm `b` đầu tiên, rồi `a` đầu tiên sau nó...

**Bài 2**
- Mức 1: chỉ cần ghi `j` **ở ranh giới cuối mỗi khối**.
- Mức 2: khối 1 chỉ khớp được `c` (chữ cuối). Khối 2 bắt đầu khi đang chờ `b`.
- Mức 3: bạn sẽ thấy `j = 1` xuất hiện lại sau khối 3, lần đầu là sau khối 1.

**Bài 3**
- Mức 1: chép khung `count_copies`, bọc thêm một vòng `for _ in range(n1)` bên ngoài.
- Mức 2: `j = 0` phải đặt **trước** cả hai vòng lặp.
- Mức 3: đừng quên `// n2` ở cuối.

**Bài 4**
- Mức 1: `seen[j]` nên lưu cả `k` và `count` lúc đó để tính `d`.
- Mức 2: `full, r = divmod(n1 - k, L)`.
- Mức 3: `for _ in range(r): j, c = run_block(j); count += c`.

**Bài 5**
- Mức 1: mỗi lần "bắt đầu đọc lại `source` từ đầu" là thêm một subsequence.
- Mức 2: nếu một lượt `source` không tăng được `i` thì chắc chắn có ký tự lạ — kiểm tra bằng `set` từ đầu cho gọn.
- Mức 3: khung giống hệt `trace_simulate`, nhưng dừng khi khớp hết **một** bản `target`.

**Bài 6**
- Mức 1: số tầng cần là số bit của `n1` (`n1.bit_length()`).
- Mức 2: kiểm tra bit `t` bằng `(n1 >> t) & 1`.
- Mức 3: thứ tự ghép các bước nhảy không quan trọng **về kết quả** vì mọi khối giống nhau — miễn là mỗi bước bắt đầu từ `j` hiện tại.

## 20. Đáp án tham khảo

### Đáp án Bài 1

1. `["ab", 3] = "ababab"`.
2. Có. Khớp tham lam trên `"abab"` (chỉ số `0..3`): `a` → chỉ số 0, `a` → chỉ số 2, `b` → chỉ số 3. Giữ `{0, 2, 3}`, xóa chỉ số 1.
3. Không: trong `"ab"` chữ `b` đứng sau `a`, không thể đảo thứ tự.
4. `"ababab"`: `b`(1) `a`(2) → bản 1; `b`(3) `a`(4) → bản 2; `b`(5) rồi hết chuỗi. `K = 2`, đáp án `2 // 1 = 2`. **Cả hai bản đều vắt qua ranh giới khối** (`b` của khối trước + `a` của khối sau).

### Đáp án Bài 2

| Sau `k` khối | `j` | `count` | Hành động |
|---:|---:|---:|---|
| 0 | 0 | 0 | `seen = {0: 0}` |
| 1 | 1 | 0 | khớp `c` → `seen[1] = 1` |
| 2 | 2 | 0 | khớp `b` → `seen[2] = 2` |
| 3 | 1 | 1 | khớp `a` (xong bản 1), rồi `c` của chính khối 3 → `j = 1` **đã thấy** |

`p = 1`, `k = 3`, `L = 2`, `d = counts[3] - counts[1] = 1`. Còn `10 - 3 = 7` khối: `divmod(7, 2) = (3, 1)`. Khối lẻ cho `counts[2] - counts[1] = 0`. `K = 1 + 3·1 + 0 = 4`, đáp án `4 // 2 = 2`.

In [34]:
# Kiểm tra Bài 1
print(repr("ab" * 3), is_subsequence("aab", "abab"), is_subsequence("ba", "ab"), count_copies("ab" * 3, "ba"))
# Kiểm tra Bài 2
_ = trace_cycle("abc", 10, "cba", 2)

'ababab' True False 2
bắt đầu : j = 0, count = 0, seen = {0: 0}
sau khối 1: j = 1, count = 0, counts = [0, 0]
  ghi seen[1] = 1 -> seen = {0: 0, 1: 1}
sau khối 2: j = 2, count = 0, counts = [0, 0, 0]
  ghi seen[2] = 2 -> seen = {0: 0, 1: 1, 2: 2}
sau khối 3: j = 1, count = 1, counts = [0, 0, 0, 1]
  j = 1 đã thấy sau 1 khối -> p = 1, k = 3, L = 2, d = 1
  còn 7 khối: full = 3 chu kỳ, r = 1 khối lẻ (cho thêm counts[2] - counts[1] = 0)
  K = 1 + 3*1 + 0 = 4  -> đáp án = 4 // 2 = 2


In [35]:
# Đáp án Bài 3
def answer_simulate(s1: str, n1: int, s2: str, n2: int) -> int:
    j = 0
    count = 0
    for _ in range(n1):
        for ch in s1:
            if ch == s2[j]:
                j += 1
                if j == len(s2):
                    count += 1
                    j = 0
    return count // n2


assert answer_simulate("acb", 4, "ab", 2) == 2
assert answer_simulate("ba", 2, "ab", 1) == 1
assert answer_simulate("abab", 9, "bbaa", 2) == 2
print("Bài 3 PASS")

Bài 3 PASS


In [36]:
# Đáp án Bài 4
def answer_cycle(s1: str, n1: int, s2: str, n2: int) -> int:
    if not set(s2) <= set(s1):
        return 0
    L2 = len(s2)

    def run_block(j):
        c = 0
        for ch in s1:
            if ch == s2[j]:
                j += 1
                if j == L2:
                    c += 1
                    j = 0
        return j, c

    j, count = 0, 0
    seen = {0: (0, 0)}                  # j -> (k, count sau k khối)
    for k in range(1, n1 + 1):
        j, c = run_block(j)
        count += c
        if j in seen:
            p, count_p = seen[j]
            L, d = k - p, count - count_p
            full, r = divmod(n1 - k, L)
            count += full * d
            for _ in range(r):          # mô phỏng r khối lẻ (r < L <= L2)
                j, c = run_block(j)
                count += c
            return count // n2
        seen[j] = (k, count)
    return count // n2


assert answer_cycle("acb", 4, "ab", 2) == 2
assert answer_cycle("abab", 9, "bbaa", 1) == 5
assert answer_cycle("abc", 10**6, "d", 1) == 0
random.seed(44)
for _ in range(2000):
    a = "".join(random.choice("abc") for _ in range(random.randint(1, 6)))
    b = "".join(random.choice("abc") for _ in range(random.randint(1, 5)))
    n1, n2 = random.randint(1, 30), random.randint(1, 5)
    assert answer_cycle(a, n1, b, n2) == repetitions_bruteforce(a, n1, b, n2), (a, n1, b, n2)
print("Bài 4 PASS (kể cả 2000 test ngẫu nhiên)")

Bài 4 PASS (kể cả 2000 test ngẫu nhiên)


In [37]:
# Đáp án Bài 5 — Shortest Way to Form String
def answer_shortest_way(source: str, target: str) -> int:
    if not set(target) <= set(source):
        return -1
    i = 0                     # vị trí chờ trong target
    blocks = 0
    while i < len(target):
        blocks += 1           # bắt đầu một subsequence mới = một lượt source
        for ch in source:
            if i < len(target) and ch == target[i]:
                i += 1
    return blocks


assert answer_shortest_way("abc", "abcbc") == 2
assert answer_shortest_way("abc", "acdbc") == -1
assert answer_shortest_way("xyz", "xzyxz") == 3
print("Bài 5 PASS")

Bài 5 PASS


### Giải thích đáp án Bài 6

Tầng `t` của bảng trả lời câu hỏi: "đang chờ `s2[j]`, đi thêm **`2^t` khối** thì chờ ở đâu và ghép thêm bao nhiêu bản?". Tầng `t` ghép từ hai lần tầng `t - 1`, nên lập được `T = n1.bit_length()` tầng trong `O(T · L2)`.

Ví dụ `n1 = 9 = 8 + 1` (nhị phân `1001`): bắt đầu `j = 0`, nhảy tầng 0 (1 khối), rồi nhảy tầng 3 (8 khối) — tổng 9 khối.

Độ phức tạp: lập bảng `next_j` `O(L1 · L2)`, các tầng `O(L2 · log n1)`, truy vấn `O(log n1)`. Ưu điểm so với chu kỳ: trả lời được **nhiều giá trị `n1` khác nhau** nhanh mà không phải tìm lại chu kỳ.

In [38]:
# Đáp án Bài 6 — binary lifting
def answer_binary_lifting(s1: str, n1: int, s2: str, n2: int) -> int:
    next_j, gain = build_table(s1, s2)
    jump, add = [next_j], [gain]
    L2 = len(s2)
    for t in range(1, n1.bit_length()):
        pj, pa = jump[-1], add[-1]
        jump.append([pj[pj[j]] for j in range(L2)])
        add.append([pa[j] + pa[pj[j]] for j in range(L2)])
    j = count = 0
    for t in range(n1.bit_length()):
        if (n1 >> t) & 1:
            count += add[t][j]
            j = jump[t][j]
    return count // n2


assert answer_binary_lifting("acb", 4, "ab", 2) == 2
assert answer_binary_lifting("abab", 9, "bbaa", 2) == 2
assert answer_binary_lifting("abc", 10**6, "cba", 1) == repetitions_table_walk("abc", 10**6, "cba", 1)
random.seed(66)
for _ in range(2000):
    a = "".join(random.choice("abc") for _ in range(random.randint(1, 6)))
    b = "".join(random.choice("abc") for _ in range(random.randint(1, 5)))
    n1, n2 = random.randint(1, 40), random.randint(1, 5)
    assert answer_binary_lifting(a, n1, b, n2) == repetitions_bruteforce(a, n1, b, n2), (a, n1, b, n2)
print("Bài 6 PASS (kể cả 2000 test ngẫu nhiên)")

Bài 6 PASS (kể cả 2000 test ngẫu nhiên)


## 21. Template ngắn dễ nhớ

```python
def getMaxRepetitions(s1, n1, s2, n2):
    if not set(s2) <= set(s1): return 0          # ký tự lạ -> 0
    j = count = 0
    seen, counts = {0: 0}, [0]                   # j ở ranh giới -> khối; count sau k khối
    for k in range(1, n1 + 1):
        for ch in s1:                            # đi một khối, khớp tham lam
            if ch == s2[j]:
                j += 1
                if j == len(s2): count += 1; j = 0
        counts.append(count)
        if j in seen:                            # lặp trạng thái -> nhảy chu kỳ
            p = seen[j]
            full, r = divmod(n1 - k, k - p)
            return (count + full * (count - counts[p]) + counts[p + r] - counts[p]) // n2
        seen[j] = k
    return count // n2
```

Câu thần chú: **"Xóa ký tự = subsequence = hai con trỏ tham lam. Trạng thái giữa các khối chỉ là `j`. `j` lặp thì nhảy: trọn vòng + khối lẻ. Cuối cùng chia `n2`."**

## 22. Checklist trước khi submit

- Đã hiểu `[str2, m] = [s2, n2 · m]` và trả về **`K // n2`** (chia lấy phần nguyên, không phải `/`)?
- Có dùng **subsequence** (hai con trỏ), không nhầm sang `in` / `.count()` của substring?
- `j` **không bị reset** ở đầu mỗi khối? (Test `("ba", 2, "ab", 1) → 1` và `("ab", 7, "ababa", 1) → 2`.)
- Trạng thái trong `seen` chỉ là **`j`**, không kèm `count`?
- Khởi tạo `seen = {0: 0}`, `counts = [0]` và hiểu `counts[k]` là count **sau** `k` khối?
- Khi nhảy chu kỳ có cộng **cả `r` khối lẻ** (`counts[p + r] - counts[p]`)? (Test `("abab", 9, "bbaa", 1)` phải ra `5`.)
- Có nhánh `return count // n2` cho trường hợp `n1` nhỏ, chưa kịp lặp?
- Đã xử lý `s2` có ký tự không có trong `s1` → `0`?
- Không xây `s1 * n1` trong bản nộp; độ phức tạp `O(len(s1) · len(s2))`?
- Đã chạy 2 ví dụ đề + ca `n1 = n2 = 10^6` + đối chiếu ngẫu nhiên với brute force?